In [4]:
# ==========================================
# 셀 1: 라이브러리 로드 및 환경 변수 설정
# ==========================================
import pandas as pd
import numpy as np
import os

# 파일 경로 설정 (사용자 지정 경로)
DATA_PATH = '/Users/judetak/Desktop/산공카르텔경진대회/2025_생활인구 data/생활인구_LOCAL_RESD_202310_202510.csv'
GRID_PATH = '/Users/judetak/Desktop/산공카르텔경진대회/격자_250m_4326.csv'
INTERMEDIATE_SAVE_PATH = '/Users/judetak/Desktop/산공카르텔경진대회/중간저장_전처리완료.csv'
FINAL_SAVE_PATH = '/Users/judetak/Desktop/산공카르텔경진대회/최종본_격자병합완료.csv'

# 컬럼명 명시 (데이터셋에 맞게 필요시 수정하세요)
COL_DATE = '일자'
COL_TIME = '시간'
COL_GRID = '250M격자'
COL_POP  = '생활인구합계'

print('✅ 환경 설정 완료')


✅ 환경 설정 완료


In [2]:


# ==========================================
# 셀 2: 원본 데이터 로드
# ==========================================
print('🔍 원본 데이터 로드 중...')
full_df = pd.read_csv(DATA_PATH)
print(f'✅ 로드 완료: {full_df.shape[0]:,}행 × {full_df.shape[1]}열')



🔍 원본 데이터 로드 중...
✅ 로드 완료: 194,035,039행 × 5열


In [6]:

# ==========================================
# 셀 3: [핵심] 격자별 누락 시간 생성 + 중앙값 대체 + 이상치 제거 + 보간
# ==========================================
print('🛠️ 격자별 [누락 시간 복원 -> 이상치 제거 -> EWMA 보간] 일괄 처리 중...')

results = []
grouped = full_df.groupby(COL_GRID)
total = len(grouped)

for i, (grid, chunk) in enumerate(grouped):
    if i % 100 == 0:
        print(f'  진행: {i:,} / {total:,} 격자 처리 중...')

    chunk = chunk.copy()
    
    # --------------------------------------------------
    # [STEP 1] 비는 시간 확인 및 다른 일자의 해당 시간대 중앙값 대체
    # --------------------------------------------------
    # 해당 격자에 존재하는 일자들 추출
    dates_in_grid = chunk[COL_DATE].unique()
    
    # 해당 격자의 (존재하는 일자 × 0~23시)의 완벽한 뼈대(Index) 생성
    mux = pd.MultiIndex.from_product(
        [[grid], dates_in_grid, range(24)], 
        names=[COL_GRID, COL_DATE, COL_TIME]
    )
    chunk = chunk.drop_duplicates(subset=[COL_GRID, COL_DATE, COL_TIME])
    # 뼈대에 맞춰 데이터프레임 재정렬 (비어있던 시간대는 인구수가 NaN으로 생성됨)
    chunk = chunk.set_index([COL_GRID, COL_DATE, COL_TIME]).reindex(mux).reset_index()
    
    # 시간대별 중앙값 계산 및 누락된 부분 채우기
    time_medians = chunk.groupby(COL_TIME)[COL_POP].transform('median')
    chunk[COL_POP] = chunk[COL_POP].fillna(time_medians)
    
    # (예외 처리) 만약 특정 시간대가 전 기간에 걸쳐 아예 없어서 중앙값도 NaN이라면 0으로 채우기
    chunk[COL_POP] = chunk[COL_POP].fillna(0)


    # --------------------------------------------------
    # [STEP 2 & 3] 이상치 제거 + EWMA 보간 (제공해주신 최적화 로직)
    # --------------------------------------------------
    chunk = chunk.sort_values(by=[COL_DATE, COL_TIME])
    orig = chunk[COL_POP].copy()

    time_grouped = chunk.groupby(COL_TIME)[COL_POP]
    
    # 각 시간대별 하위 10%, 상위 10%
    p10 = time_grouped.transform(lambda x: x.quantile(0.10))
    p90 = time_grouped.transform(lambda x: x.quantile(0.90))
    p_range = p90 - p10
    
    lo = p10 - 3 * p_range
    hi = p90 + 3 * p_range
    
    outlier = (chunk[COL_POP] < lo) | (chunk[COL_POP] > hi)
    chunk.loc[outlier, COL_POP] = np.nan

    # EWMA + bfill 보간
    chunk[COL_POP] = chunk[COL_POP].fillna(
        chunk[COL_POP].ewm(span=3, min_periods=1, ignore_na=True).mean()
    ).bfill()

    # 전체 기간 NaN이면 원본(중앙값으로 채워둔 원본) 복원
    if chunk[COL_POP].isna().all():
        chunk[COL_POP] = orig

    results.append(chunk)

print('🔄 전처리 결과 병합 중...')
full_df_processed = pd.concat(results, ignore_index=True)
print(f'✅ 전처리 완료! 잔여 결측치: {full_df_processed[COL_POP].isna().sum():,}')



🛠️ 격자별 [누락 시간 복원 -> 이상치 제거 -> EWMA 보간] 일괄 처리 중...
  진행: 0 / 8,886 격자 처리 중...
  진행: 100 / 8,886 격자 처리 중...
  진행: 200 / 8,886 격자 처리 중...
  진행: 300 / 8,886 격자 처리 중...
  진행: 400 / 8,886 격자 처리 중...
  진행: 500 / 8,886 격자 처리 중...
  진행: 600 / 8,886 격자 처리 중...
  진행: 700 / 8,886 격자 처리 중...
  진행: 800 / 8,886 격자 처리 중...
  진행: 900 / 8,886 격자 처리 중...
  진행: 1,000 / 8,886 격자 처리 중...
  진행: 1,100 / 8,886 격자 처리 중...
  진행: 1,200 / 8,886 격자 처리 중...
  진행: 1,300 / 8,886 격자 처리 중...
  진행: 1,400 / 8,886 격자 처리 중...
  진행: 1,500 / 8,886 격자 처리 중...
  진행: 1,600 / 8,886 격자 처리 중...
  진행: 1,700 / 8,886 격자 처리 중...
  진행: 1,800 / 8,886 격자 처리 중...
  진행: 1,900 / 8,886 격자 처리 중...
  진행: 2,000 / 8,886 격자 처리 중...
  진행: 2,100 / 8,886 격자 처리 중...
  진행: 2,200 / 8,886 격자 처리 중...
  진행: 2,300 / 8,886 격자 처리 중...
  진행: 2,400 / 8,886 격자 처리 중...
  진행: 2,500 / 8,886 격자 처리 중...
  진행: 2,600 / 8,886 격자 처리 중...
  진행: 2,700 / 8,886 격자 처리 중...
  진행: 2,800 / 8,886 격자 처리 중...
  진행: 2,900 / 8,886 격자 처리 중...
  진행: 3,000 / 8,886 격자 처리 중...
  진행: 3,100

In [7]:
# ==========================================
# 셀 3.5: 데이터 무결성 검증 (결측치 및 시간대 누락 확인)
# ==========================================
print('🔍 전처리 데이터 무결성 검증 시작...\n')

# 1. 결측치(NaN) 잔여 여부 확인
print('[1] 결측치 확인')
missing_counts = full_df_processed.isna().sum()

if missing_counts.sum() == 0:
    print('  ✅ 완벽합니다! 데이터프레임 내에 결측치(NaN)가 단 하나도 없습니다.')
else:
    print('  ⚠️ 주의: 아래 컬럼에 결측치가 남아있습니다.')
    print(missing_counts[missing_counts > 0])


# 2. 모든 (격자, 일자) 조합에 24시간(0~23시)이 모두 존재하는지 확인
print('\n[2] 24시간(0~23시) 누락 여부 확인')
# 격자와 일자별로 묶었을 때 데이터 행의 개수(시간의 개수)가 24개인지 확인합니다.
time_counts = full_df_processed.groupby([COL_GRID, COL_DATE])[COL_TIME].count()
missing_hours_groups = time_counts[time_counts != 24]

if len(missing_hours_groups) == 0:
    print('  ✅ 완벽합니다! 모든 격자의 모든 일자에 24시간 데이터가 꽉 채워져 있습니다.')
else:
    print(f'  ⚠️ 주의: 24시간이 꽉 채워지지 않은 (격자, 일자) 조합이 {len(missing_hours_groups):,}개 있습니다.')
    print('  [예시 보기]')
    print(missing_hours_groups.head())

print('\n✅ 검증 완료!')

🔍 전처리 데이터 무결성 검증 시작...

[1] 결측치 확인
  ⚠️ 주의: 아래 컬럼에 결측치가 남아있습니다.
Unnamed: 0    548322
dtype: int64

[2] 24시간(0~23시) 누락 여부 확인
  ✅ 완벽합니다! 모든 격자의 모든 일자에 24시간 데이터가 꽉 채워져 있습니다.

✅ 검증 완료!


In [8]:

# ==========================================
# 셀 4: 중간 저장 (선택 사항 - 안전을 위해 권장)
# ==========================================
# 'Unnamed'가 포함된 모든 찌꺼기 컬럼 삭제
full_df_processed = full_df_processed.loc[:, ~full_df_processed.columns.str.contains('^Unnamed')]

print(f'✅ 불필요한 컬럼 삭제 완료! 현재 컬럼: {full_df_processed.columns.tolist()}')
print('💾 전처리 완료 데이터 중간 저장 중...')
full_df_processed.to_csv(INTERMEDIATE_SAVE_PATH, index=False, encoding='utf-8-sig')
print(f'✅ 중간 저장 완료: {INTERMEDIATE_SAVE_PATH}')



✅ 불필요한 컬럼 삭제 완료! 현재 컬럼: ['250M격자', '일자', '시간', '생활인구합계']
💾 전처리 완료 데이터 중간 저장 중...
✅ 중간 저장 완료: /Users/judetak/Desktop/산공카르텔경진대회/중간저장_전처리완료.csv


In [9]:

# ==========================================
# 셀 5: 위치(Geospatial) 격자 데이터와 병합 및 최종 저장
# ==========================================
print('🔗 250M 격자 공간 데이터 병합 중...')
df_grid = pd.read_csv(GRID_PATH)

# '250M격자'와 'CELL_ID' 기준으로 병합 (Left Join)
final_df = pd.merge(
    full_df_processed, 
    df_grid, 
    left_on=COL_GRID, 
    right_on='CELL_ID', 
    how='left'
)

print('💾 최종본 저장 중...')
final_df.to_csv(FINAL_SAVE_PATH, index=False, encoding='utf-8-sig')
print(f'🎉 모든 작업 완료! 최종 데이터 크기: {final_df.shape[0]:,}행 × {final_df.shape[1]}열')
print(f'📂 최종 저장 경로: {FINAL_SAVE_PATH}')

🔗 250M 격자 공간 데이터 병합 중...
💾 최종본 저장 중...
🎉 모든 작업 완료! 최종 데이터 크기: 156,228,024행 × 11열
📂 최종 저장 경로: /Users/judetak/Desktop/산공카르텔경진대회/최종본_격자병합완료.csv


In [2]:
import pandas as pd

In [8]:
final_df = pd.read_csv('/Users/judetak/Desktop/산공카르텔경진대회/생활인구 데이터_격자 병합 완료.csv')

In [9]:
node_df = pd.read_csv('/Users/judetak/Desktop/산공카르텔경진대회/서울시 미세먼지 관측소_격자.csv')

In [11]:
node_df.head()

,item,mangName,year,addr,stationNam,lat,lon,fid,CELL_ID
0,"SO2, CO, O3, NO2, PM10, PM2.5",도시대기,1995,서울 중구 덕수궁길 15 시청서소문별관 3동,중구,37.564639,126.975961,218,다사53505175
1,"SO2, CO, O3, NO2, PM10, PM2.5",도로변대기,1996,서울 용산구 한강대로 405 (서울역 앞),한강대로,37.549389,126.971519,291,다사53255000
2,"SO2, CO, O3, NO2, PM10, PM2.5",도시대기,1997,"서울 종로구 종로35가길 19 종로5,6가 동 주민센터",종로구,37.572025,127.005028,9335,다사56255250
3,"SO2, CO, O3, NO2, PM10, PM2.5",도로변대기,1994,서울 중구 청계천로 184 (청계천4가사거리 남강빌딩 앞),청계천로,37.568650,126.998083,9684,다사55505225
4,"SO2, CO, O3, NO2, PM10, PM2.5",도로변대기,2008,서울 종로구 종로 169 (종묘주차장 앞),종로,37.570633,126.996783,9684,다사55505225


In [6]:
final_df.head()

,250M격자,일자,시간,생활인구합계,fid,CELL_ID,CELL_X,CELL_Y,GID,lon,lat
69530088,다사54504050,20231001,0,255.84,1.0,다사54504050,954625.0,1940625.0,다사54ba40ba,126.985479,37.464854
69530089,다사54504050,20231001,1,267.69,1.0,다사54504050,954625.0,1940625.0,다사54ba40ba,126.985479,37.464854
69530090,다사54504050,20231001,2,271.09,1.0,다사54504050,954625.0,1940625.0,다사54ba40ba,126.985479,37.464854
69530091,다사54504050,20231001,3,271.47,1.0,다사54504050,954625.0,1940625.0,다사54ba40ba,126.985479,37.464854
69530092,다사54504050,20231001,4,248.85,1.0,다사54504050,954625.0,1940625.0,다사54ba40ba,126.985479,37.464854


In [12]:
df2 = final_df[final_df['CELL_ID'].isin(node_df['CELL_ID'].unique())].copy()
print(f'필터링 전: {final_df.shape[0]:,}행')
print(f'필터링 후: {df2.shape[0]:,}행')

필터링 전: 156,228,024행
필터링 후: 712,704행


In [15]:

df2 = df2.merge(node_df[['CELL_ID', 'stationNam']], on='CELL_ID', how='left')
print(f'merge 완료: {df2.shape[0]:,}행 × {df2.shape[1]}열')
df2.head()


merge 완료: 730,992행 × 12열


,250M격자,일자,시간,생활인구합계,fid,CELL_ID,CELL_X,CELL_Y,GID,lon,lat,stationNam
0,다사40255150,20231001,0,99.41,5969.0,다사40255150,940375.0,1951625.0,다사40ab51ba,126.823453,37.563189,공항대로
1,다사40255150,20231001,1,96.77,5969.0,다사40255150,940375.0,1951625.0,다사40ab51ba,126.823453,37.563189,공항대로
2,다사40255150,20231001,2,92.55,5969.0,다사40255150,940375.0,1951625.0,다사40ab51ba,126.823453,37.563189,공항대로
3,다사40255150,20231001,3,89.52,5969.0,다사40255150,940375.0,1951625.0,다사40ab51ba,126.823453,37.563189,공항대로
4,다사40255150,20231001,4,87.76,5969.0,다사40255150,940375.0,1951625.0,다사40ab51ba,126.823453,37.563189,공항대로


In [16]:
df2.to_csv('미세먼지 관측소 노드별 생활인구.csv')